# 04 — Deep Learning Models: Load & Evaluate
Load pre-trained LSTM, STGCN, DCRNN checkpoints and evaluate on test data.

**No training** — loads saved `.pt` files from `results/models/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import torch
import matplotlib.pyplot as plt
from src import config
from src.config import set_seed, get_device
from src.data_loader import prepare_dataset
from src.graph_builder import build_graph
from src.evaluate import evaluate_predictions, print_results, compare_models
set_seed(42); device = get_device()

## 1. Prepare Data & Graph

In [ ]:
DATASET = 'METR-LA'  # Change to 'PEMS-BAY' to evaluate that dataset
filepath = config.DATASETS[DATASET]['path']

data = prepare_dataset(filepath, batch_size=config.BATCH_SIZE)
mean, std = data['mean'], data['std']
num_sensors = data['splits']['train'][0].shape[2]

train_end = int(len(data['raw_data']) * config.TRAIN_RATIO)
graph = build_graph(data['raw_data'][:train_end],
                    sigma=config.GRAPH_SIGMA, epsilon=config.GRAPH_EPSILON)

test_X = torch.FloatTensor(data['splits']['test'][0]).to(device)
test_Y = data['splits']['test'][1]
print(f"Dataset: {DATASET} | Sensors: {num_sensors} | Test samples: {len(test_X)}")

## 2. Load & Evaluate LSTM

In [ ]:
from src.models.lstm_model import LSTMModel

lstm = LSTMModel(num_sensors, config.SEQ_LEN, config.PRED_LEN,
                 config.LSTM_HIDDEN, config.LSTM_LAYERS, config.LSTM_DROPOUT).to(device)

ckpt = f'../results/models/lstm_{DATASET}_best.pt'
lstm.load_state_dict(torch.load(ckpt, weights_only=True, map_location=device))
lstm.eval()
print(f"Loaded: {ckpt}")
print(f"Parameters: {sum(p.numel() for p in lstm.parameters()):,}")

In [ ]:
preds_lstm = []
with torch.no_grad():
    for i in range(0, len(test_X), config.BATCH_SIZE):
        batch = test_X[i:i+config.BATCH_SIZE]
        preds_lstm.append(lstm(batch).cpu().numpy())
preds_lstm = np.concatenate(preds_lstm)

r_lstm = evaluate_predictions(preds_lstm, test_Y, mean, std)
print_results(r_lstm, 'LSTM')

## 3. Load & Evaluate STGCN

In [ ]:
from src.models.stgcn import STGCN

stgcn = STGCN(num_sensors, config.SEQ_LEN, config.PRED_LEN,
              K=config.STGCN_K, channels=config.STGCN_CHANNELS).to(device)

ckpt = f'../results/models/stgcn_{DATASET}_best.pt'
stgcn.load_state_dict(torch.load(ckpt, weights_only=True, map_location=device))
stgcn.eval()
print(f"Loaded: {ckpt}")
print(f"Parameters: {sum(p.numel() for p in stgcn.parameters()):,}")

In [ ]:
cheb = [torch.FloatTensor(p).to(device) for p in graph['cheb_polys']]
preds_stgcn = []
with torch.no_grad():
    for i in range(0, len(test_X), config.BATCH_SIZE):
        batch = test_X[i:i+config.BATCH_SIZE]
        preds_stgcn.append(stgcn(batch, cheb).cpu().numpy())
preds_stgcn = np.concatenate(preds_stgcn)

r_stgcn = evaluate_predictions(preds_stgcn, test_Y, mean, std)
print_results(r_stgcn, 'STGCN')

## 4. Load & Evaluate DCRNN

In [ ]:
from src.models.dcrnn import DCRNN

n_sup = len(graph['diffusion_supports'])
dcrnn = DCRNN(num_sensors, n_sup, config.SEQ_LEN, config.PRED_LEN,
              config.DCRNN_HIDDEN, config.DCRNN_LAYERS).to(device)

ckpt = f'../results/models/dcrnn_{DATASET}_best.pt'
dcrnn.load_state_dict(torch.load(ckpt, weights_only=True, map_location=device))
dcrnn.eval()
print(f"Loaded: {ckpt}")
print(f"Parameters: {sum(p.numel() for p in dcrnn.parameters()):,}")

In [ ]:
supports = [torch.FloatTensor(s).to(device) for s in graph['diffusion_supports']]
preds_dcrnn = []
with torch.no_grad():
    for i in range(0, len(test_X), config.BATCH_SIZE):
        batch = test_X[i:i+config.BATCH_SIZE]
        preds_dcrnn.append(dcrnn(batch, supports).cpu().numpy())
preds_dcrnn = np.concatenate(preds_dcrnn)

r_dcrnn = evaluate_predictions(preds_dcrnn, test_Y, mean, std)
print_results(r_dcrnn, 'DCRNN')

## 5. Compare All Deep Models

In [ ]:
all_results = {'LSTM': r_lstm, 'STGCN': r_stgcn, 'DCRNN': r_dcrnn}
compare_models(all_results, DATASET)

In [ ]:
# GNN improvement over LSTM
lstm_mae = r_lstm['overall']['MAE']
for name, r in [('STGCN', r_stgcn), ('DCRNN', r_dcrnn)]:
    gnn_mae = r['overall']['MAE']
    imp = (lstm_mae - gnn_mae) / lstm_mae * 100
    print(f"{name} vs LSTM: {imp:+.1f}% MAE {'(better)' if imp > 0 else '(worse)'}")

## 6. Prediction vs Ground Truth

In [ ]:
from src.data_loader import denormalize_data

all_preds = {'LSTM': preds_lstm, 'STGCN': preds_stgcn, 'DCRNN': preds_dcrnn}
colors = {'LSTM':'#2ecc71', 'STGCN':'#3498db', 'DCRNN':'#9b59b6'}

for sensor_idx in [0, 50]:
    if sensor_idx >= num_sensors:
        continue
    fig, ax = plt.subplots(figsize=(14, 5))
    T = 288  # 1 day

    gt_denorm = denormalize_data(test_Y[:T, 0, sensor_idx], mean[sensor_idx], std[sensor_idx])
    ax.plot(gt_denorm, color='black', linewidth=2, label='Ground Truth', alpha=0.9)

    for name, pred in all_preds.items():
        pred_denorm = denormalize_data(pred[:T, 0, sensor_idx], mean[sensor_idx], std[sensor_idx])
        ax.plot(pred_denorm, color=colors[name], linewidth=1.5, label=name, alpha=0.7)

    ax.set_title(f'15-min Prediction — Sensor {sensor_idx} ({DATASET})', fontsize=13)
    ax.set_xlabel('Time Step (5-min intervals)')
    ax.set_ylabel('Speed (mph)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'../results/plots/{DATASET}_pred_sensor{sensor_idx}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Per-Sensor Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, pred) in enumerate(all_preds.items()):
    pred_d = pred * std[np.newaxis, np.newaxis, :] + mean[np.newaxis, np.newaxis, :]
    gt_d = test_Y * std[np.newaxis, np.newaxis, :] + mean[np.newaxis, np.newaxis, :]
    sensor_mae = np.mean(np.abs(pred_d - gt_d), axis=(0, 1))
    axes[idx].bar(range(len(sensor_mae)), sensor_mae, color=colors[name], alpha=0.7, width=1.0)
    axes[idx].set_title(f'{name} — Per-Sensor MAE', fontsize=12)
    axes[idx].set_xlabel('Sensor Index'); axes[idx].set_ylabel('MAE')
    axes[idx].axhline(sensor_mae.mean(), color='red', linestyle='--', alpha=0.5)
plt.suptitle(f'Spatial Error Distribution — {DATASET}', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'../results/plots/{DATASET}_spatial_error.png', dpi=150, bbox_inches='tight')
plt.show()